<img src="./images/DLI_Header.png" style="width: 400px;">

# 5. Generating Math Problems and Solutions with NeMo Curator

In this notebook, we will demonstrate how to use **NVIDIA NeMo Curator** to generate a set of math problems and their solutions on a specific topic.

The generated outputs will then be evaluated using the **Llama-3.1-nemotron-70b-reward model** to ensure quality and relevance.

**[5.1 NeMo Curator OpenAI Client](#5.1-NeMo-Curator-OpenAI-Client)<br>**
**[5.2 Math problems generation in English and Spanish](#5.2-Math-problems-generation-in-English-and-Spanish)<br>**
**[5.3 Combining subtopics and math problems generation to create a diverse set of math problems in English and Spanish](#5.3-Combining-subtopics-and-math-problems-generation-to-create-a-diverse-set-of-math-problems-in-English-and-Spanish)<br>**


---

## Connecting to the NVIDIA API Catalog

NeMo Curator supports connecting to [OpenAI API](https://github.com/openai/openai-python?tab=readme-ov-file#openai-python-api-library) compatible services and [NeMo Deploy](https://docs.nvidia.com/nemo-framework/user-guide/latest/deployingthenemoframeworkmodel.html#use-nemo-export-and-deploy-module-apis-to-run-inference) services.

In this notebook, we rely on the `build.nvidia.com` API endpoints. You can use this same flow with a model deployed as an NVIDIA NIM for LLMs which can be found [here](https://github.com/NVIDIA/NeMo-Curator/blob/main/docs/user-guide/syntheticdata.rst#connecting-to-an-llm-service).

Your environment already has an NVIDIA API key installed for you. For work outside of this workshop environment, please see the instructions below for how to obtain your own free NVIDIA API key.

### Obtaining Your Own NVIDIA API Key

If you would like an NVIDIA API key for your own work outside this workshop environment, you can generate one for free using the following steps:

1. Login (or sign up) through [build.nvidia.com](https://build.nvidia.com/explore/discover).
2. Click the `Get API Key` button available on the the `Llama 3.1 Nemotron 70B Reward` page, found [here](https://build.nvidia.com/nvidia/llama-3_1-nemotron-70b-reward).

---

## 5.1 NeMo Curator OpenAI Client

We will begin by initializing NeMo Curator's `OpenAI Client`.

Please note that this step is identical to the process outlined in the previous notebook.

In [8]:
import os

from nemo_curator import OpenAIClient
from nemo_curator.synthetic import NemotronGenerator
from openai import OpenAI

# Initialize OpenAI's client with the NVIDIA API endpoint and the API key.
openai_client = OpenAI(
    # Outside this workshop environment you would set NVIDIA_BASE_URL to "https://integrate.api.nvidia.com/v1".
    base_url=os.environ["NVIDIA_BASE_URL"],
    api_key=os.environ["NVIDIA_API_KEY"],
)

# Initialize NeMo Curator's OpenAIClient by passing the OpenAI client instance.
# This wraps the OpenAI client to provide additional functionality specific to NeMo Curator.
curator_openai_client = OpenAIClient(openai_client)

# Create an instance of NemotronGenerator, which facilitates synthetic data generation.
generator = NemotronGenerator(curator_openai_client)

# Model used to generate syntethic data.
model = "meta/llama3-70b-instruct"
model_kwargs = {
    "temperature": 0.1,
    "top_p": 0.9,
    "max_tokens": 1024,
}

## 5.2 Math problems generation in English and Spanish
We will generate math problems based on a specific subtopic. Instead of using the `convert_response_to_yaml_list` method, we will create and use **a custom parser**.

In order to do it, we will slightly modify the default `MATH_PROBLEM_GENERAL_PROMPT_TEMPLATE`, and will use the following instead:

```python
    "Create {n_openlines} diverse mathematics problems related to the topic '{topic}' " \
    "or solvable using concepts from '{topic}'. Provide your response as a numbered list, " \
    "and include both the problem description and its solution. Format your response as follows: " \
    "((###1###)) >>>Problem<<<: [Description of the first problem]. >>>Solution<<<: [Solution to the first problem].\n" \
    "((###2###)) >>>Problem<<<: [Description of the second problem]. >>>Solution<<<: [Solution to the second problem].\n" \
    "Only include the problems and their solutions—no additional text."
```


In [9]:
math_prompt_template_english = (
    "Create {n_openlines} diverse mathematics problems related to the topic '{topic}' "
    "or solvable using concepts from '{topic}'. Provide your response as a numbered list, "
    "and include both the problem description and its solution. Format your response as follows: "
    "((###1###)) >>>Problem<<<: [Description of the first problem]. >>>Solution<<<: [Solution to the first problem].\n"
    "((###2###)) >>>Problem<<<: [Description of the second problem]. >>>Solution<<<: [Solution to the second problem].\n"
    "Only include the problems and their solutions—no additional text."
)

n_problems = 3
topic = "Algebra"

llm_response = generator.generate_math_problem(
    model=model,
    topic=topic,
    n_openlines=n_problems,
    prompt_template=math_prompt_template_english,
)

print(llm_response[0])

((###1###)) >>>Problem<<<: Solve for x in the equation 2x + 5 = 11.

>>>Solution<<<: Subtract 5 from both sides of the equation to get 2x = 11 - 5, which simplifies to 2x = 6. Then, divide both sides by 2 to get x = 6/2, which simplifies to x = 3.

((###2###)) >>>Problem<<<: Find the value of x in the equation x^2 + 4x - 8 = 0.

>>>Solution<<<: Factor the left-hand side of the equation to get (x + 2)(x - 4) = 0. This tells us that either (x + 2) = 0 or (x - 4) = 0. Solving for the first factor, we get x + 2 = 0 --> x = -2. Solving for the second factor, we get x - 4 = 0 --> x = 4. Therefore, the values of x are -2 and 4.

((###3###)) >>>Problem<<<: If Sally can paint a house in 6 hours, and John can paint the same house in 4 hours, how long will it take for both of them to paint the house together?

>>>Solution<<<: Let's say the rate at which Sally paints the house is 1/6 of the house per hour, and the rate at which John paints the house is 1/4 of the house per hour. When they work tog

In this case, we will use a customer parser to clean our list of questions/answers.

In [10]:
import re
from typing import List, Tuple


def parse_math_llm_response(
    llm_response: str, tag_replacements: dict
) -> List[Tuple[str, str]]:
    # Extract the first and second tags from the tag_replacements dictionary
    first_tag, second_tag = list(tag_replacements.keys())[:2]

    # Regular expression to match each problem-solution block using the first and second tags
    pattern = rf"{re.escape(first_tag)}(.*?){re.escape(second_tag)}(.*?)(?=\(\(###|\Z)"

    # Find all matches using re.DOTALL to handle multiline content
    matches = re.findall(pattern, llm_response, re.DOTALL)

    # Clean up and replace tags in extracted problems and solutions
    problem_solution_pairs = []
    for problem, solution in matches:
        # Replace tags with their corresponding replacements
        for tag, replacement in tag_replacements.items():
            problem = problem.replace(tag, replacement).strip()
            solution = solution.replace(tag, replacement).strip()

        problem_solution_pairs.append((problem.strip(), solution.strip()))

    return problem_solution_pairs


def generate_math_problems(
    model: str,
    model_kwargs: dict,
    topic: str,
    n_problems: int,
    math_prompt_template: str,
    tag_replacements: dict,
    n_retries: int = 5,
) -> List[Tuple[str, str]]:
    math_problems = []
    for _ in range(n_retries):
        try:
            llm_response = generator.generate_math_problem(
                model=model,
                topic=topic,
                n_openlines=n_problems,
                prompt_template=math_prompt_template,
            )
            math_problems = parse_math_llm_response(
                llm_response=llm_response[0], tag_replacements=tag_replacements
            )
            break
        except Exception as e:
            print(f"Hit: {e}, Retrying...")

    return math_problems


# We will use these tags to find the problem and solution
tag_replacements_english = {
    ">>>Problem<<<:": "Problem:",
    ">>>Solution<<<:": "Solution:",
}

# Example usage
math_problems_english = generate_math_problems(
    model=model,
    model_kwargs=model_kwargs,
    topic=topic,
    n_problems=n_problems,
    math_prompt_template=math_prompt_template_english,
    tag_replacements=tag_replacements_english,
)

# Print each problem-solution pair
for idx, (problem, solution) in enumerate(math_problems_english):
    print(f"Problem: {problem}")
    print(f"Solution: {solution}\n")

Problem: Solve for x in the equation 2x + 5 = 11.
Solution: Subtract 5 from both sides of the equation to get 2x = 11 - 5, which simplifies to 2x = 6. Then, divide both sides by 2 to get x = 6/2, which simplifies to x = 3.

Problem: If Sally can paint a house in 6 hours, and John can paint the same house in 4 hours, how long will it take for both of them to paint the house together?
Solution: Let's use the concept of work and rate to solve this problem. Let the total work be 1 house. Sally's rate is 1/6 of a house per hour, and John's rate is 1/4 of a house per hour. Their combined rate is 1/6 + 1/4 = (2+3)/12 = 5/12 of a house per hour. To find the time it takes for them to paint the house together, divide the total work by their combined rate: 1 house / (5/12) = 12/5 hours, which simplifies to 2.4 hours.

Problem: Solve the system of linear equations: x + 2y = 7, 3x - 2y = 1.
Solution: We can solve this system using substitution or elimination. Let's use elimination. Multiply the fir

Now it’s time to do the same in Spanish. To accomplish this, we simply need to adjust the prompt as follows:

In [11]:
math_prompt_template_spanish = (
    "Crea {n_openlines} problemas de matemáticas diversos relacionados con el tema '{topic}' "
    "o que se puedan resolver utilizando conceptos de '{topic}'. Proporciona tu respuesta como una lista numerada, "
    "e incluye tanto la descripción del problema como su solución. Formatea tu respuesta de la siguiente manera: "
    "((###1###)) >>>Problema<<<: [Descripción del primer problema]. >>>Solución<<<: [Solución del primer problema].\n"
    "((###2###)) >>>Problema<<<: [Descripción del segundo problema]. >>>Solución<<<: [Solución del segundo problema].\n"
    "Incluye únicamente los problemas y sus soluciones, sin texto adicional. Responde usando sólo el idioma Español."
)

# We will use these tags to find the problem and solution
tag_replacements_spanish = {
    ">>>Problema<<<:": "Problema:",
    ">>>Solución<<<:": "Solución:",
}

math_problems_spanish = generate_math_problems(
    model=model,
    model_kwargs=model_kwargs,
    topic=topic,
    n_problems=n_problems,
    math_prompt_template=math_prompt_template_spanish,
    tag_replacements=tag_replacements_spanish,
)

# Print each problem-solution pair
for idx, (problem, solution) in enumerate(math_problems_spanish):
    print(f"Problema: {problem}")
    print(f"Solución: {solution}\n")

Problema: Resolver la ecuación cuadrada x^2 + 5x + 6 = 0.
Solución: Para resolver esta ecuación, podemos factorizar el trinomio: x^2 + 5x + 6 = (x + 3)(x + 2) = 0. Luego, podemos igualar cada factor a cero y resolver para x: x + 3 = 0 => x = -3, x + 2 = 0 => x = -2. Por lo tanto, las soluciones son x = -3 y x = -2.

Problema: Hallar el valor de x en la expresión algebraica 2x + 5 = 11.
Solución: Para resolver esta ecuación, podemos restar 5 de ambos lados: 2x = 11 - 5 => 2x = 6. Luego, podemos dividir ambos lados entre 2: x = 6/2 => x = 3. Por lo tanto, el valor de x es 3.

Problema: Simplificar la expresión algebraica (2x^2 + 3x - 1) / (x + 2) cuando x ≠ -2.
Solución: Para simplificar esta expresión, podemos factorizar el numerador: 2x^2 + 3x - 1 = (2x - 1)(x + 1). Luego, podemos cancelar el factor común (x + 2) entre el numerador y el denominador: ((2x - 1)(x + 1)) / (x + 2) = 2x - 1. Por lo tanto, la expresión simplificada es 2x - 1.



## 5.3 Combining subtopics and math problems generation to create a diverse set of math problems in English and Spanish

Now it's time to bring together everything we've covered so far. To achieve this, we will begin by generating a list of `n` math subtopics and then create `m` pairs of problem-solution sets for each subtopic.

Let's begin with the English language!

In [12]:
from utils import generate_subtopics


def math_pipeline(
    generator: NemotronGenerator,
    model: str,
    model_kwargs: dict,
    topic: str,
    n_subtopics: int,
    n_problems: int,
    subtopics_prompt_template: str,
    math_prompt_template: str,
    tag_replacements: dict,
    n_retries: int = 5,
) -> List[Tuple[str, str]]:

    # Generate n math subtopic
    print(f"Generating {n_subtopics} subtopics for {topic}...")
    subtopics = generate_subtopics(
        generator=generator,
        model=model,
        model_kwargs=model_kwargs,
        macro_topic=topic,
        n_subtopics=n_subtopics,
        prompt_template=subtopics_prompt_template,
    )

    # Print the subtopics
    for subtopic in subtopics:
        print(f"\t{subtopic}")

    # Convert tag_replacements values to list, to use in the next step
    tag_replacements_values = list(tag_replacements.values())

    # Generate n problems per subtopic
    math_problems = []
    for subtopic in subtopics:
        print(f"\nGenerating {n_problems} {subtopic} problem-solution pair(s)...")
        try:
            llm_response = generate_math_problems(
                model=model,
                model_kwargs=model_kwargs,
                topic=subtopic,
                n_problems=n_problems,
                math_prompt_template=math_prompt_template,
                tag_replacements=tag_replacements,
            )

            # Print each problem-solution pair
            for idx, (problem, solution) in enumerate(llm_response):
                print(f"{tag_replacements_values[0]} {problem}")
                print(f"{tag_replacements_values[1]} {solution}\n")

            math_problems.extend(llm_response)
        except Exception as e:
            print(f"Hit: {e}")

    return math_problems


# Define a prompt template for generating subtopics in English.
subtopics_prompt_template_english = (
    "Generate {n_subtopics} topics that cover various aspects of {macro_topic}. "
    "Your response should only be a list of topics, as diverse as possible. Do not include explanations or additional text. For example: "
    "1. Food and drinks. \n2. Technology.\n"
)

topic_english = "Math"
n_subtopics = 3
n_problems = 3

# Run the pipeline for English
math_problems_english = math_pipeline(
    generator=generator,
    model=model,
    model_kwargs=model_kwargs,
    topic=topic_english,
    n_subtopics=n_subtopics,
    n_problems=n_problems,
    subtopics_prompt_template=subtopics_prompt_template_english,
    math_prompt_template=math_prompt_template_english,
    tag_replacements=tag_replacements_english,
)

Generating 3 subtopics for Math...
	Fractals and Self-Similarity
	Cryptography and Number Theory
	Chaos Theory and Dynamical Systems

Generating 3 Fractals and Self-Similarity problem-solution pair(s)...
Problem: A Sierpinski triangle has an area of 16 square units. If the area of the largest triangle in the first iteration is 4 square units, what is the area of the largest triangle in the second iteration?
Solution: The area of the largest triangle in the first iteration is 1/4 of the original area, so the area of the largest triangle in the second iteration is 1/4 of the area of the largest triangle in the first iteration, which is (1/4)(4) = 1 square unit.

Problem: A Menger sponge has a volume of 27 cubic units. If the volume of each of the 20 smaller cubes in the first iteration is 1 cubic unit, what is the volume of each of the 20^2 smaller cubes in the second iteration?
Solution: The volume of each of the 20 smaller cubes in the first iteration is 1/20 of the original volume, so

In [13]:
# Define a prompt template for generating subtopics in Spanish.
subtopics_prompt_template_spanish = (
    "Genera {n_subtopics} temas amplios que abarquen diversos aspectos de {macro_topic}. "
    "Tu respuesta debe ser únicamente una lista de temas. "
    "Cada tema es un elemento de la lista. No incluyas subtemas ni explicaciones ni texto adicional. "
    "Por ejemplo: 1. Comida y bebidas. \n2. Tecnología.\n"
)

# Run the pipeline for Spanish
topic_spanish = "Matemáticas"
math_problems_spanish = math_pipeline(
    generator=generator,
    model=model,
    model_kwargs=model_kwargs,
    topic=topic_spanish,
    n_subtopics=n_subtopics,
    n_problems=n_problems,
    subtopics_prompt_template=subtopics_prompt_template_spanish,
    math_prompt_template=math_prompt_template_spanish,
    tag_replacements=tag_replacements_spanish,
)

Generating 3 subtopics for Matemáticas...
	Números y Operaciones
	Espacio y Forma
	Cambio y Variación

Generating 3 Números y Operaciones problem-solution pair(s)...
Problema: Un librero tiene 250 libros en su tienda. Regala 30 libros a una biblioteca y vende 45 libros a un cliente. ¿Cuántos libros le quedan al librero?
Solución: 250 - 30 - 45 = 175 libros.

Problema: Un número es 5 veces mayor que otro número. Si el número menor es 12, ¿cuál es el número mayor?
Solución: Si el número menor es 12, el número mayor es 12 x 5 = 60.

Problema: Un estudiante tiene 18 monedas de 50 céntimos y 25 monedas de 10 céntimos. ¿Cuánto dinero tiene en total?
Solución: 18 x 50 = 900 céntimos. 25 x 10 = 250 céntimos. En total tiene 900 + 250 = 1150 céntimos, que es igual a 11.50 euros.


Generating 3 Espacio y Forma problem-solution pair(s)...
Problema: Un cubo tiene una arista de 5 cm. ¿Cuál es el volumen del cubo?
Solución: El volumen del cubo es V = a³, donde a es la arista del cubo. En este caso, a

**Exercise:** Using `nvidia/llama-3.1-nemotron-70b-reward`, filter the generated examples to retain only those that meet a certain quality threshold.

In [25]:
from typing import List, Tuple, Dict
import re

# ------------------------------------------------------------
# Compose messages exactly like the Nemotron Reward expects
# ------------------------------------------------------------
def compose_messages(user_content: str, assistant_content: str) -> List[Dict[str, str]]:
    return [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]


# ------------------------------------------------------------
# Call Nemotron Reward model and parse reward score
# ------------------------------------------------------------
def get_reward(messages: List[Dict[str, str]]) -> float:
    response = openai_client.chat.completions.create(
        model="nvidia/llama-3.1-nemotron-70b-reward",
        messages=messages,
        temperature=0.0,
        max_tokens=10,
    )

    content = response.choices[0].message.content.strip()

    # Expected formats:
    # "reward:-17.125"
    # "-17.125"
    match = re.search(r"-?\d+(\.\d+)?", content)
    if not match:
        raise ValueError(f"Invalid reward output: {content}")

    return float(match.group())


# ------------------------------------------------------------
# Evaluate math problem + solution pairs
# ------------------------------------------------------------
def evaluate_math_problems(
    model: str,
    math_problems: List[Tuple[str, str]],
    min_threshold: float = -20.0,
    max_retries: int = 5,
):
    retained: List[Tuple[str, str, float]] = []
    discarded: List[Tuple[str, str, float]] = []

    for index, (problem, solution) in enumerate(math_problems):
        for attempt in range(max_retries):
            try:
                messages = compose_messages(
                    user_content=problem,
                    assistant_content=solution,
                )

                reward = get_reward(messages)

                if reward >= min_threshold:
                    print(f"Reward: {reward:.3f} | Problem {index + 1} RETAINED")
                    retained.append((problem, solution, reward))
                else:
                    print(f"Reward: {reward:.3f} | Problem {index + 1} DISCARDED")
                    discarded.append((problem, solution, reward))

                break  # sucesso → sai do retry

            except Exception as e:
                print(
                    f"Erro no problema {index + 1}, tentativa {attempt + 1}: {e}"
                )
                if attempt == max_retries - 1:
                    raise RuntimeError(
                        f"Falhou no problema {index + 1} após {max_retries} tentativas"
                    )

    return retained, discarded


# ------------------------------------------------------------
# Run evaluation
# ------------------------------------------------------------
retained_math_problems, discarded_math_problems = evaluate_math_problems(
    model="nvidia/llama-3.1-nemotron-70b-reward",
    math_problems=math_problems_english,
    min_threshold=-20.0,
)

print(f"\nRetained: {len(retained_math_problems)}")
print(f"Discarded: {len(discarded_math_problems)}")


Reward: -17.125 | Problem 1 RETAINED
Reward: -21.375 | Problem 2 DISCARDED
Reward: -17.750 | Problem 3 RETAINED
Reward: -11.125 | Problem 4 RETAINED
Reward: -11.000 | Problem 5 RETAINED
Reward: -19.125 | Problem 6 RETAINED
Reward: -13.812 | Problem 7 RETAINED
Reward: -14.688 | Problem 8 RETAINED
Reward: -15.500 | Problem 9 RETAINED

Retained: 8
Discarded: 1


---
<h2 style="color:green;">Congratulations!</h2>

In this notebook, you used Nemo Curator to generate math problems along with their solutions on a specific topic in both English and Spanish.

<img src="./images/DLI_Header.png" style="width: 400px; float: right;">